In [1]:
# ==================== EDGE AI CHALLENGE 2026 - TRAINING ====================
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import time

# ====================== THAY ĐƯỜNG DẪN Ở ĐÂY ======================
# Bạn thêm data sau, chỉ cần sửa 2 dòng dưới
DATA_PATH = "/kaggle/input/edge-ai-challenge-2026"   # ←←← THAY ĐƯỜNG DẪN CỦA BẠN
TRAIN_DIR = f"C:\DUT\Ki 2\Edge AI\dataset/train"                     # thư mục chứa 10 folder 0-9
TEST_DIR  = f"C:\DUT\Ki 2\Edge AI\dataset/test"                      # chứa ảnh 00000.jpg, 00001.jpg...

# ====================== DATA LOADER ======================
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(32, 32),
    batch_size=128,
    class_mode='categorical',
    color_mode='rgb',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(32, 32),
    batch_size=128,
    class_mode='categorical',
    color_mode='rgb',
    subset='validation',
    shuffle=False
)

print("✅ Số class:", train_generator.num_classes)  # phải = 10

# ====================== MÔ HÌNH SIÊU NHẸ (48k params) ======================
model = models.Sequential([
    layers.Conv2D(8, (3,3), activation='relu', input_shape=(32,32,3), padding='same'),
    layers.MaxPooling2D(2,2),
    
    layers.Conv2D(16, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),
    
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D(2,2),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("📊 Tổng tham số:", model.count_params())  # ~48.000

# ====================== TRAINING ======================
history = model.fit(
    train_generator,
    epochs=50,
    validation_data=val_generator,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4)
    ]
)

# Save model gốc
model.save("traffic_sign_model.h5")
print("✅ Đã lưu traffic_sign_model.h5")

# ====================== ĐÁNH GIÁ ======================
val_loss, val_acc = model.evaluate(val_generator)
print(f"🎯 Validation Accuracy: {val_acc*100:.2f}%")

# ====================== QUANTIZE → .tflite (int8) ======================
# Representative dataset cho calibration
def representative_data_gen():
    for _ in range(100):
        img, _ = next(iter(train_generator))
        yield [img[:50]]   # 50 ảnh đủ để calibrate

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("model_quant_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ Đã tạo model_quant_int8.tflite (int8)")

# ====================== DATASHEET (copy paste vào phần nộp) ======================
print("\n" + "="*60)
print("📋 DATASHEET MÔ HÌNH")
print("="*60)
print(f"Số tham số          : {model.count_params():,}")
print(f"Input               : 32×32×3 RGB")
print(f"Output              : 10 class (0-9)")
print(f"Framework           : TensorFlow 2.x")
print(f"Model gốc           : traffic_sign_model.h5")
print(f"Model tối ưu        : model_quant_int8.tflite")
print(f"Val Accuracy        : {val_acc*100:.2f}%")
print(f"Size .tflite        : {len(tflite_model)/1024:.1f} KB")
print(f"Thời gian suy luận (CPU) ước tính: ~12-18ms trên ESP32-S3 (sẽ test chính thức)")
print("="*60)

# Lưu datasheet
with open("datasheet.txt", "w") as f:
    f.write(f"Số tham số: {model.count_params():,}\n")
    f.write(f"Accuracy: {val_acc*100:.2f}%\n")
    f.write("Model size (tflite): ~45KB\n")

<>:14: SyntaxWarning: invalid escape sequence '\D'
<>:15: SyntaxWarning: invalid escape sequence '\D'
<>:14: SyntaxWarning: invalid escape sequence '\D'
<>:15: SyntaxWarning: invalid escape sequence '\D'
C:\Users\ngong\AppData\Local\Temp\ipykernel_13544\1638965032.py:14: SyntaxWarning: invalid escape sequence '\D'
  TRAIN_DIR = f"C:\DUT\Ki 2\Edge AI\dataset/train"                     # thư mục chứa 10 folder 0-9
C:\Users\ngong\AppData\Local\Temp\ipykernel_13544\1638965032.py:15: SyntaxWarning: invalid escape sequence '\D'
  TEST_DIR  = f"C:\DUT\Ki 2\Edge AI\dataset/test"                      # chứa ảnh 00000.jpg, 00001.jpg...


Found 1600 images belonging to 10 classes.
Found 400 images belonging to 10 classes.
✅ Số class: 10


c:\Users\ngong\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 8)      │           224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 32)       │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

Saved artifact at 'C:\Users\ngong\AppData\Local\Temp\tmpks4n7hta'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1357063707664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063708432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063709776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063707472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063709200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063710544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063707856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063710928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063711312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1357063712848: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\ngong\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


✅ Đã tạo model_quant_int8.tflite (int8)

📋 DATASHEET MÔ HÌNH
Số tham số          : 39,514
Input               : 32×32×3 RGB
Output              : 10 class (0-9)
Framework           : TensorFlow 2.x
Model gốc           : traffic_sign_model.h5
Model tối ưu        : model_quant_int8.tflite
Val Accuracy        : 63.00%
Size .tflite        : 46.5 KB
Thời gian suy luận (CPU) ước tính: ~12-18ms trên ESP32-S3 (sẽ test chính thức)


C:\Users\ngong\AppData\Local\Temp\ipykernel_13544\1638965032.py:14: SyntaxWarning: invalid escape sequence '\D'
  TRAIN_DIR = f"C:\DUT\Ki 2\Edge AI\dataset/train"                     # thư mục chứa 10 folder 0-9
C:\Users\ngong\AppData\Local\Temp\ipykernel_13544\1638965032.py:15: SyntaxWarning: invalid escape sequence '\D'
  TEST_DIR  = f"C:\DUT\Ki 2\Edge AI\dataset/test"                      # chứa ảnh 00000.jpg, 00001.jpg...


UnicodeEncodeError: 'charmap' codec can't encode character '\u1ed1' in position 1: character maps to <undefined>